# Chapter 6 &mdash; Changing the Model: Context, Width, and Training Time

**Concept 15 of the Chapter 6 decomposition:** *Changing the Model: Context, Width, and How Long You Train*

One language, fixed. Now move the five numbers at the top and see which knob actually helps.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter6-DFAOps/Concept-Changing-The-Model/Concept-Changing-The-Model.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Now the language is held still and the **model** is the variable.

Five numbers sit at the top of Karpathy's config:

```python
block_size = context_length    # how many symbols it can see
vocab_size = 2                 # how many tokens exist
n_layer    = 4                 # attention blocks
n_head     = 4                 # attention heads per block
n_embd     = 16                # width of the representation
```

plus how many iterations you train for.

Raising `context_length` from 3 to 4 doubles the state space, 8 nodes to 16, and the
picture grows with it. Whether it grows *better* depends on the language &mdash; that
is the question to sit with.

`n_embd` gives the model more room per state. Training longer lets the loss settle.
Neither buys memory that the context window does not have, and finding that out by
hand is worth more than being told.

## 2. Definitions

### The model, the picture, the training loop

In [ ]:
#@title minimal GPT implementation in PyTorch  (Andrej Karpathy)
#
# Read it, change it, break it.  This is the whole model: an embedding, a
# few attention blocks, a linear head.  Nothing here knows about automata.
""" super minimal decoder-only gpt """

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k ,v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # manual implementation of attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.c_proj(y)
        return y

class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.nonlin = nn.GELU()

    def forward(self, x):
        x = self.c_fc(x)
        x = self.nonlin(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    # these are default GPT-2 hyperparameters
    block_size: int = 1024
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    bias: bool = False

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %d" % (sum(p.nelement() for p in self.parameters()),))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0) # shape (1, t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (1, t, n_embd)
        x = tok_emb + pos_emb
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x[:, -1, :]) # note: only returning logits at the last time step (-1), output is 2D (b, vocab_size)
        return logits

&nbsp;

In [ ]:
# --- Karpathy's picture: the model AS a finite-state machine -------------
from graphviz import Digraph

def all_possible(n, k):
    # every list of k elements, each in range(n)
    if k == 0:
        yield []
    else:
        for i in range(n):
            for c in all_possible(n, k - 1):
                yield [i] + c

def plot_model(gpt, title=''):
    dot = Digraph(comment='baby GPT', engine='circo')
    if title:
        dot.attr(label=title, labelloc='t', fontsize='12')
    for xi in all_possible(gpt.config.vocab_size, gpt.config.block_size):
        x = torch.tensor(xi, dtype=torch.long)[None, ...]
        y = nn.functional.softmax(gpt(x), dim=-1)[0].tolist()
        here = ''.join(str(d) for d in xi)
        dot.node(here)
        for t in range(gpt.config.vocab_size):
            nxt = ''.join(str(d) for d in (xi[1:] + [t]))
            dot.edge(here, nxt, label='%d(%.0f%%)' % (t, y[t] * 100),
                     color='#b3251f' if t == 0 else '#1a53c0')
    return dot

&nbsp;

In [ ]:
# --- the training data: strings the DFA accepts, run together ------------
from functools import reduce

def corpus_from(D, upto=400):
    strings = [nthnumeric(i, ['0', '1']) for i in range(upto)]
    good = [s for s in strings if accepts_dfa(D, s)]
    return good, list(map(int, reduce(lambda a, b: a + b, good)))

def make_XY(seq, context_length):
    X, Y = [], []
    for i in range(len(seq) - context_length):
        X.append(seq[i:i + context_length])
        Y.append(seq[i + context_length])
    return (torch.tensor(X, dtype=torch.long),
            torch.tensor(Y, dtype=torch.long))

def train_gpt(gpt, X, Y, iters=200, lr=1e-3, every=20):
    optimizer = torch.optim.AdamW(gpt.parameters(), lr=lr, weight_decay=1e-1)
    losses = []
    for i in range(iters):
        logits = gpt(X)
        loss = F.cross_entropy(logits, Y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        losses.append(loss.item())
        if i % every == 0 or i == iters - 1:
            print(i, loss.item())
    return losses

### One language, held fixed

In [ ]:
# --- the language, as a Jove DFA  --  CHANGE THIS ------------------------
# No two 1s in a row.  A good first language because the constraint is
# LOCAL: after a 1, a 1 is forbidden, and three bits of context is plenty
# to see that.  Try ENDS01 below once you have seen this one work.
NO11 = md2mc('''DFA
IF : 0 -> IF
IF : 1 -> F1
F1 : 0 -> IF
F1 : 1 -> D
D  : 0|1 -> D
''')

ENDS01 = md2mc('''DFA
I  : 0 -> S0
I  : 1 -> I
S0 : 0 -> S0
S0 : 1 -> F
F  : 0 -> S0
F  : 1 -> I
''')

LANG = NO11                      # <-- change me
dotObj_dfa(min_dfa(LANG), FuseEdges=True)

<!-- nav-strip -->

---

&larr;&nbsp;[Ch6&nbsp;14.&nbsp;Changing the Language: Which Ones Does It Pick Up?](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter6-DFAOps/Concept-Changing-The-Language/Concept-Changing-The-Language.ipynb) &nbsp;&middot;&nbsp; [**Chapter 6** index](https://github.com/ganeshutah/Jove/blob/master/Chapter6-DFAOps/README.md) &nbsp;&middot;&nbsp; [Ch7&nbsp;1.&nbsp;Nondeterminism as Forking Tokens, and as Guessing](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter7-NFA/Concept-Forking-Tokens/Concept-Forking-Tokens.ipynb)&nbsp;&rarr;

---

## 3. Tests

A helper with every knob exposed.

In [ ]:
def run(D, context_length=3, n_embd=16, n_layer=4, n_head=4,
        iters=200, lr=1e-3, seed=1337, label=''):
    good, seq = corpus_from(D)
    X, Y = make_XY(seq, context_length)
    config = GPTConfig(block_size=context_length, vocab_size=2,
                       n_layer=n_layer, n_head=n_head, n_embd=n_embd,
                       bias=False)
    torch.manual_seed(seed)
    gpt = GPT(config)
    losses = train_gpt(gpt, X, Y, iters=iters, lr=lr, every=iters)
    print('%-28s %2d states, final loss %.4f'
          % (label, 2 ** context_length, losses[-1]))
    return gpt, losses

**How long to train.** The same model, stopped at four different points.

In [ ]:
for it in (10, 50, 200, 800):
    run(ENDS01, iters=it, label='iters=%d' % it)

**How much context.** Each extra symbol doubles the states.

In [ ]:
for k in (2, 3, 4):
    run(ENDS01, context_length=k, label='context_length=%d' % k)

**How wide.** More room per state.

In [ ]:
for w in (8, 16, 64):
    run(ENDS01, n_embd=w, label='n_embd=%d' % w)

Pick your favourite setting and look at the picture it gives.

In [ ]:
gpt, losses = run(ENDS01, context_length=3, n_embd=32, iters=400,
                  label='context 3, width 32, 400 iters')
plot_model(gpt, 'context 3, width 32, 400 iterations')

Sixteen states, if you want to see one.

In [ ]:
# --- sample from the model, exactly as Karpathy does --------------------
def sample(gpt, start, steps=24):
    xi = list(start)
    full = xi.copy()
    for _ in range(steps):
        x = torch.tensor(xi, dtype=torch.long)[None, ...]
        probs = nn.functional.softmax(gpt(x), dim=-1)
        t = torch.multinomial(probs[0], num_samples=1).item()
        xi = xi[1:] + [t]
        full.append(t)
    return ''.join(map(str, full))


gpt4, _ = run(ENDS01, context_length=4, iters=400, label='context_length=4')
plot_model(gpt4, 'context length 4 -- sixteen states')

## 4. Exercises


1. Which of the three knobs moved the loss most? Was it the one you expected?
2. `n_layer` and `n_head` are untouched above. Try `n_layer=1`. How much does it cost?
3. At `context_length=4` the model has sixteen states and the DFA still has three.
   What are the extra thirteen doing?
4. Train for 5000 iterations. Does the loss keep falling, and does the picture keep
   changing? What does the gap between those two answers tell you?
5. Set `lr=1.0`. Read the loss printout and describe what went wrong.
6. Go back to the parity DFA from the previous notebook and try *every* knob on it.
   Nothing works. Write down the one-sentence reason.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter6-DFAOps/Concept-Changing-The-Model')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')